In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/adnansaqib1212/final-data-et-of-fraud-detection/final_data.csv


In [2]:
df =  pd.read_csv('/kaggle/input/datasets/adnansaqib1212/final-data-et-of-fraud-detection/final_data.csv')
df.head()

,amt,gender,city_pop,is_fraud,hour,merchant_rate,age,distance_km,categ_fraud_rate
0,3.78,M,3032,0,8,0.142857,62,38.148475,0.160541
1,166.03,M,13835,0,19,0.070423,59,73.759559,0.052608
2,46.02,M,970,0,5,0.128205,32,96.165888,0.070712
3,7.27,F,31394,0,20,0.265625,29,85.240398,0.275685
4,67.66,M,83,0,5,0.108527,68,30.376435,0.108593


In [3]:
df['gender'] = df['gender'].map({'M':1,'F':0})

In [4]:
df.head()

,amt,gender,city_pop,is_fraud,hour,merchant_rate,age,distance_km,categ_fraud_rate
0,3.78,1,3032,0,8,0.142857,62,38.148475,0.160541
1,166.03,1,13835,0,19,0.070423,59,73.759559,0.052608
2,46.02,1,970,0,5,0.128205,32,96.165888,0.070712
3,7.27,0,31394,0,20,0.265625,29,85.240398,0.275685
4,67.66,1,83,0,5,0.108527,68,30.376435,0.108593


In [5]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

In [6]:
x = df.drop(columns=['is_fraud'])
y = df['is_fraud']

In [9]:
# Master dictionary containing both LightGBM and XGBoost grids
param_grids = {
    "xgboost": {
        "n_estimators": [100, 200, 300, 500, 800],
        "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
        "max_depth": [3, 4, 5, 6, 8, 10],
        "min_child_weight": [1, 3, 5, 7],
        "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        "gamma": [0, 0.1, 0.2, 0.3],
        "reg_alpha": [0, 0.01, 0.1, 1, 10],
        "reg_lambda": [0.1, 1, 5, 10],
    },
    "lightgbm": {
        "n_estimators": [100, 200, 300, 500, 800],
        "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
        "num_leaves": [15, 31, 63, 127, 255],
        "max_depth": [-1, 3, 5, 7, 10],
        "min_child_samples": [10, 20, 30, 50],
        "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
        "reg_alpha": [0, 0.01, 0.1, 1, 10],
        "reg_lambda": [0, 0.01, 0.1, 1, 10],
    },
}
models = {
    'xgboost' : XGBClassifier(tree_method="hist",
    device="cuda", 
    verbosity=0, 
    random_state=42,),
    'lightgbm': LGBMClassifier(device="gpu", 
    verbosity=-1, 
    random_state=42,)
}

In [10]:
best_models = {}

for model_name, model in models.items():
    print(f"Currently training: {model_name}")
    current_params = param_grids[model_name]

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=current_params,
        n_iter=10,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        random_state=42,
    )

    random_search.fit(x, y)

    best_models[model_name] = random_search.best_estimator_
    print(f"Best {model_name} ROC-AUC Score: {random_search.best_score_:.4f}\n")

Currently training: xgboost
Best xgboost ROC-AUC Score: 0.9985

Currently training: lightgbm


11 warning generated11.
 warning warning warning generated generated.
 generated.
.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning gen

Best lightgbm ROC-AUC Score: 0.9985



In [17]:
from sklearn.metrics import confusion_matrix

y_pred = best_models["lightgbm"].predict(x)
cm = confusion_matrix(y, y_pred)
print(cm)

[[50000     0]
 [    0  7506]]


In [19]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = best_models["lightgbm"].predict(x)

print("=== Classification Report ===")
print(classification_report(y, y_pred, digits=4))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y, y_pred)
print(cm)

=== Classification Report ===
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000     50000
           1     1.0000    1.0000    1.0000      7506

    accuracy                         1.0000     57506
   macro avg     1.0000    1.0000    1.0000     57506
weighted avg     1.0000    1.0000    1.0000     57506


=== Confusion Matrix ===
[[50000     0]
 [    0  7506]]


In [20]:
import pandas as pd

test_data = {
    "amt": [12.50, 450.00, 89.99, 5.20, 1200.50],
    "gender": [1, 0, 1, 0, 1],  # 1: Male, 0: Female
    "city_pop": [15000, 500, 250000, 4200, 85000],
    "hour": [14, 2, 18, 11, 3],
    "merchant_rate": [0.085200, 0.452100, 0.112000, 0.051000, 0.620000],
    "age": [45, 23, 37, 61, 29],
    "distance_km": [12.450123, 115.820145, 5.120491, 42.891023, 180.450112],
    "categ_fraud_rate": [0.065120, 0.385000, 0.089120, 0.042100, 0.512000],
}

X_test_sample = pd.DataFrame(test_data)


y_pred_sample = best_models["lightgbm"].predict(X_test_sample)
y_pred_proba = best_models["lightgbm"].predict_proba(X_test_sample)[:, 1]

X_test_sample["predicted_fraud"] = y_pred_sample
X_test_sample["fraud_probability"] = y_pred_proba
print(X_test_sample[["amt", "predicted_fraud", "fraud_probability"]])

       amt  predicted_fraud  fraud_probability
0    12.50                1           0.586325
1   450.00                0           0.021342
2    89.99                0           0.000004
3     5.20                0           0.000005
4  1200.50                1           0.993511
